In [ ]:
import pandas as pd

MMS_CSV = "mms_predictions.csv"
SPEECHBRAIN_CSV = "speechbrain_predictions.csv"
WHISPER_SMALL_CSV = "whisper_small_predictions.csv"
WHISPER_MED_CSV = "whisper_medium_predictions.csv"

mms = pd.read_csv(MMS_CSV)
sb = pd.read_csv(SPEECHBRAIN_CSV)
ws = pd.read_csv(WHISPER_SMALL_CSV)
wm = pd.read_csv(WHISPER_MED_CSV)


In [ ]:
base = wm.copy()

base["split_clean"] = (
    base["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

base["speech_true"] = (
    base["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

base["speech_type_clean"] = (
    base["speech_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

def parse_languages(x):
    if pd.isna(x):
        return set()

    langs = {
        str(lang).strip().lower()
        for lang in str(x).split(";")
        if str(lang).strip()
    }

    langs -= {"neutral", "mixed", "unclear"}

    return langs

base["true_language_set"] = base["langs_present"].apply(parse_languages)
base["n_valid_languages"] = base["true_language_set"].map(len)

base = base[
    base["split_clean"].eq("dev")
    & base["speech_true"].eq("yes")
    & base["n_valid_languages"].eq(1)
    & base["speech_type_clean"].eq("live")
].copy()

base["true_language"] = base["true_language_set"].apply(
    lambda x: next(iter(x))
)

# Common cross-model subset: exclude Cantonese
base = base[
    ~base["true_language"].eq("cantonese")
].copy()

preds = (
    base[["file_id", "true_language"]]
    .merge(
        mms[["file_id", "mms_prediction"]].drop_duplicates("file_id"),
        on="file_id",
        how="inner"
    )
    .merge(
        sb[["file_id", "speechbrain_prediction"]].drop_duplicates("file_id"),
        on="file_id",
        how="inner"
    )
    .merge(
        ws[["file_id", "whisper_prediction"]].drop_duplicates("file_id"),
        on="file_id",
        how="inner"
    )
    .merge(
        wm[["file_id", "whisper_med_prediction"]].drop_duplicates("file_id"),
        on="file_id",
        how="inner"
    )
)

model_cols = {
    "MMS-LID-256": "mms_prediction",
    "SpeechBrain": "speechbrain_prediction",
    "Whisper-small": "whisper_prediction",
    "Whisper-medium": "whisper_med_prediction",
}

def normalise_prediction(x):
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()

    if x == "modern greek (1453-)":
        x = "greek"

    return x

for col in model_cols.values():
    preds[col] = preds[col].apply(normalise_prediction)

rows = []

for model, col in model_cols.items():

    x = preds[preds[col].ne("")].copy()

    x["true_for_scoring"] = x["true_language"]

    # reference mapping for models that use "Chinese" for Mandarin

    if model in {"SpeechBrain", "Whisper-small", "Whisper-medium"}:
        x["true_for_scoring"] = x["true_for_scoring"].replace({
            "mandarin": "chinese"
        })

    incorrect = x[
        x[col] != x["true_for_scoring"]
    ].copy()

    # count Norwegian outputs (all varieties)

    nynorsk_mask = incorrect[col].str.contains(
        r"\bnynorsk\b|norwegian nynorsk",
        case=False,
        regex=True,
        na=False
    )

    norwegian_any_mask = incorrect[col].str.contains(
        r"nynorsk|bokm[åa]l|norwegian",
        case=False,
        regex=True,
        na=False
    )

    n_errors = len(incorrect)
    n_nynorsk = int(nynorsk_mask.sum())
    n_norwegian_any = int(norwegian_any_mask.sum())

    rows.append({
        "model": model,
        "n_scored": len(x),
        "n_errors": n_errors,
        "nynorsk_errors": n_nynorsk,
        "nynorsk_pct_of_errors":
            n_nynorsk / n_errors * 100 if n_errors else 0,
        "any_norwegian_variety_errors": n_norwegian_any,
        "any_norwegian_variety_pct_of_errors":
            n_norwegian_any / n_errors * 100 if n_errors else 0
    })

results = pd.DataFrame(rows)

print(
    results.to_string(
        index=False,
        formatters={
            "nynorsk_pct_of_errors": "{:.2f}".format,
            "any_norwegian_variety_pct_of_errors": "{:.2f}".format
        }
    )
)

print("\nMOST COMMON WRONG OUTPUTS BY MODEL")

for model, col in model_cols.items():

    x = preds[preds[col].ne("")].copy()
    x["true_for_scoring"] = x["true_language"]

    if model in {"SpeechBrain", "Whisper-small", "Whisper-medium"}:
        x["true_for_scoring"] = x["true_for_scoring"].replace({
            "mandarin": "chinese"
        })

    incorrect = x[
        x[col] != x["true_for_scoring"]
    ]

    print(f"\n{model}")
    print(
        incorrect[col]
        .value_counts()
        .head(10)
        .to_string()
    )


In [ ]:
# use Whisper-medium file only for split definition

base = wm.copy()

base["split_clean"] = (
    base["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

base["speech_true"] = (
    base["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# development clips annotated as containing no speech

base = base[
    base["split_clean"].eq("dev")
    & base["speech_true"].eq("no")
].copy()

print("No-speech development clips:", len(base))

preds = (
    base[["file_id"]]
    .merge(
        mms[["file_id", "mms_prediction"]].drop_duplicates("file_id"),
        on="file_id",
        how="inner"
    )
    .merge(
        sb[["file_id", "speechbrain_prediction"]].drop_duplicates("file_id"),
        on="file_id",
        how="inner"
    )
    .merge(
        ws[["file_id", "whisper_prediction"]].drop_duplicates("file_id"),
        on="file_id",
        how="inner"
    )
    .merge(
        wm[["file_id", "whisper_med_prediction"]].drop_duplicates("file_id"),
        on="file_id",
        how="inner"
    )
)

model_cols = {
    "MMS-LID-256": "mms_prediction",
    "SpeechBrain": "speechbrain_prediction",
    "Whisper-small": "whisper_prediction",
    "Whisper-medium": "whisper_med_prediction",
}

def normalise_prediction(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

for col in model_cols.values():
    preds[col] = preds[col].apply(normalise_prediction)

rows = []

for model, col in model_cols.items():

    x = preds[preds[col].ne("")].copy()

    nynorsk_mask = x[col].str.contains(
        r"\bnynorsk\b|norwegian nynorsk",
        regex=True,
        na=False
    )

    norwegian_any_mask = x[col].str.contains(
        r"nynorsk|bokm[åa]l|norwegian",
        regex=True,
        na=False
    )

    n_scored = len(x)
    n_nynorsk = int(nynorsk_mask.sum())
    n_norwegian_any = int(norwegian_any_mask.sum())

    rows.append({
        "model": model,
        "n_no_speech_scored": n_scored,
        "nynorsk_predictions": n_nynorsk,
        "nynorsk_pct_no_speech":
            n_nynorsk / n_scored * 100 if n_scored else 0,
        "any_norwegian_variety_predictions": n_norwegian_any,
        "any_norwegian_variety_pct_no_speech":
            n_norwegian_any / n_scored * 100 if n_scored else 0
    })

results = pd.DataFrame(rows)

print("\nNORWEGIAN-VARIETY PREDICTIONS ON NO-SPEECH DEV CLIPS\n")

print(
    results.to_string(
        index=False,
        formatters={
            "nynorsk_pct_no_speech": "{:.2f}".format,
            "any_norwegian_variety_pct_no_speech": "{:.2f}".format
        }
    )
)

print("\nMOST COMMON PREDICTIONS ON NO-SPEECH CLIPS")

for model, col in model_cols.items():

    x = preds[preds[col].ne("")]

    print(f"\n{model}")
    print(
        x[col]
        .value_counts()
        .head(10)
        .to_string()
    )
